# LeetCode 249 — Group Shifted Strings bằng Python

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**
>
> Ghi chú độ khó: danh mục của repo xếp bài này vào **Easy**; LeetCode hiện ghi **Medium**.

- Link đề: https://leetcode.com/problems/group-shifted-strings/
- Độ khó (theo danh mục): **Easy** · Chủ đề: Array, Hash Table, String

Chữ ký hàm LeetCode:

```python
class Solution:
    def groupStrings(self, strings: List[str]) -> List[List[str]]:

```

**Mục tiêu học**

1. Hiểu phép **dịch (shift)** một chuỗi: mọi chữ cùng tiến 1 bước trong bảng chữ cái, `z` quay vòng về `a`.
2. Ôn `ord`/`chr`, phép `% 26` để "quay vòng", `tuple` làm key của `dict`, `defaultdict(list)`.
3. Viết baseline so từng cặp chuỗi (`O(n²)`), thấy nút thắt.
4. Tìm **khóa chuẩn hóa** (canonical key): hai chuỗi cùng nhóm ⇔ cùng khóa → gom nhóm một lần duyệt bằng hash map.
5. Biết hai kiểu khóa: dãy hiệu liên tiếp và "dịch về bắt đầu bằng `a`".

## 1. Đề bài và ví dụ

Ta có thể **dịch** một chuỗi bằng cách thay mỗi chữ bằng chữ **kế tiếp** trong bảng chữ cái, ví dụ `"abc" → "bcd"`. Chữ `z` dịch thành `a` (quay vòng), ví dụ `"az" → "ba"`.

Dịch liên tục tạo thành một **dãy dịch** (shifting sequence):

```text
"abc" -> "bcd" -> "cde" -> ... -> "xyz" -> "yza" -> "zab" -> "abc" -> ...
```

Cho mảng chuỗi `strings`, hãy **gom nhóm** các chuỗi thuộc cùng một dãy dịch. Trả kết quả theo **thứ tự bất kỳ**.

Ví dụ 1:

```text
Input : ["abc","bcd","acef","xyz","az","ba","a","z"]
Output: [["acef"],["a","z"],["abc","bcd","xyz"],["az","ba"]]
```

Ví dụ 2:

```text
Input : ["a"]
Output: [["a"]]
```

**Constraints (theo hiểu biết về đề):** `1 <= strings.length <= 200`; `1 <= strings[i].length <= 50`; `strings[i]` chỉ gồm chữ thường tiếng Anh.

Nhận xét ngay: hai chuỗi **khác độ dài** không bao giờ cùng nhóm (dịch không đổi độ dài).

## 2. Nền tảng: chữ cái ↔ số, và `% 26` để quay vòng

Đổi chữ thành số `0..25`: `ord(ch) - ord('a')`. Dịch `k` bước: `(số + k) % 26`. Đổi ngược: `chr(số + ord('a'))`.

In [1]:
def shift_char(ch: str, k: int) -> str:
    return chr((ord(ch) - ord("a") + k) % 26 + ord("a"))

print(shift_char("a", 1), shift_char("y", 1), shift_char("z", 1), shift_char("z", 3))

def shift_string(s: str, k: int) -> str:
    return "".join(shift_char(ch, k) for ch in s)

print(shift_string("abc", 1), shift_string("xyz", 1), shift_string("az", 1))

b z a c
bcd yza ba


`% 26` giữ kết quả trong `0..25`: `25 + 1 = 26`, `26 % 26 = 0` → `'a'`.

Với số âm, Python trả phần dư **không âm** khi chia cho số dương — rất tiện để tính "lùi":

In [2]:
print(-1 % 26, -27 % 26, (ord("a") - ord("b")) % 26)

25 25 25


`(ord('a') - ord('b')) % 26 = -1 % 26 = 25`: đi từ `b` tới `a` cần tiến **25** bước (quay vòng). Ở C++/Java, `-1 % 26 = -1`, phải viết `(x % 26 + 26) % 26`.

### Hai chuỗi có thuộc cùng dãy dịch không?

`s` và `t` cùng nhóm ⇔ cùng độ dài và có một số `k` (0..25) sao cho `shift_string(s, k) == t`.

In [3]:
def same_group_try_all(s: str, t: str) -> bool:
    if len(s) != len(t):
        return False
    return any(shift_string(s, k) == t for k in range(26))

print(same_group_try_all("abc", "xyz"), same_group_try_all("az", "ba"), same_group_try_all("abc", "acef"))

True True False


`any(...)` trả `True` nếu **ít nhất một** phần tử thỏa (dừng sớm).

Thử cả 26 giá trị `k` là thừa: `k` bắt buộc là khoảng cách từ `s[0]` tới `t[0]`, tức `k = (ord(t[0]) - ord(s[0])) % 26`. Chỉ cần thử đúng một `k`.

In [4]:
def same_group(s: str, t: str) -> bool:
    if len(s) != len(t):
        return False
    k = (ord(t[0]) - ord(s[0])) % 26
    return shift_string(s, k) == t

print(same_group("abc", "xyz"), same_group("az", "ba"), same_group("a", "z"), same_group("ab", "ba"))

True True True False


## 3. Cách 1 — Brute force: so mỗi chuỗi với đại diện các nhóm đã có

1. Danh sách `groups` rỗng.
2. Với mỗi chuỗi `s`: duyệt các nhóm, nếu `same_group(nhóm[0], s)` thì thêm `s` vào nhóm đó; không có nhóm nào hợp → tạo nhóm mới `[s]`.

In [5]:
def group_bruteforce(strings: list[str]) -> list[list[str]]:
    groups = []
    for s in strings:
        for g in groups:
            if same_group(g[0], s):
                g.append(s)
                break
        else:                       # for-else: chạy khi vòng for KHÔNG gặp break
            groups.append([s])
    return groups

ex1 = ["abc", "bcd", "acef", "xyz", "az", "ba", "a", "z"]
print(group_bruteforce(ex1))
print(group_bruteforce(["a"]))

[['abc', 'bcd', 'xyz'], ['acef'], ['az', 'ba'], ['a', 'z']]
[['a']]


**`for ... else`** trong Python: khối `else` chạy khi vòng `for` kết thúc **tự nhiên** (không bị `break`). Ở đây nghĩa là "không tìm thấy nhóm nào phù hợp".

### Nút thắt

Mỗi chuỗi có thể phải so với **mọi nhóm** → tệ nhất `n` nhóm × `n` chuỗi × `O(L)` mỗi lần so = `O(n² · L)`. Với `n = 200` vẫn chạy được, nhưng ta muốn cách tổng quát hơn: **gán cho mỗi chuỗi một "chữ ký" sao cho cùng nhóm ⇔ cùng chữ ký**, rồi gom bằng dict trong một lần duyệt.

In [6]:
def count_compares(strings):
    groups, compares = [], 0
    for s in strings:
        for g in groups:
            compares += 1
            if same_group(g[0], s):
                g.append(s)
                break
        else:
            groups.append([s])
    return compares

distinct = ["a" + chr(97 + i % 26) + chr(97 + i // 26) for i in range(200)]   # 200 chuỗi, mỗi chuỗi một nhóm riêng
print("số nhóm:", len(group_bruteforce(distinct)), "| số lần so sánh:", count_compares(distinct), "≈ n²/2 =", 200 * 199 // 2)

số nhóm: 200 | số lần so sánh: 19900 ≈ n²/2 = 19900


## 4. Insight — dãy **hiệu liên tiếp** không đổi khi dịch

Khi dịch cả chuỗi `k` bước, **khoảng cách giữa hai chữ cạnh nhau** không đổi (cả hai cùng tiến `k`). Vậy hiệu `(s[i] - s[i-1]) % 26` là bất biến của cả nhóm.

| chuỗi | hiệu (b-a, c-b) |
|---|---|
| `abc` | (1, 1) |
| `bcd` | (1, 1) |
| `xyz` | (1, 1) |
| `az` | (25,) |
| `ba` | (25,) vì `a - b = -1 ≡ 25` |
| `acef` | (2, 2, 1) |
| `a`, `z` | () — tuple rỗng |

Ngược lại, nếu hai chuỗi cùng độ dài có cùng dãy hiệu, dịch chuỗi này `k = t[0] - s[0]` bước sẽ khớp chữ đầu, rồi các hiệu bằng nhau kéo theo mọi chữ sau khớp. ⇒ **Cùng nhóm ⇔ cùng dãy hiệu.** (Độ dài được mã hóa sẵn: dãy hiệu có `L - 1` phần tử.)

In [7]:
def diff_key(s: str) -> tuple:
    return tuple((ord(s[i]) - ord(s[i - 1])) % 26 for i in range(1, len(s)))

for s in ex1:
    print(f"{s:<5} -> {diff_key(s)}")

abc   -> (1, 1)
bcd   -> (1, 1)
acef  -> (2, 2, 1)
xyz   -> (1, 1)
az    -> (25,)
ba    -> (25,)
a     -> ()
z     -> ()


Vì sao `tuple` mà không phải `list`? Key của dict phải **bất biến (hashable)**. `list` có thể sửa nên không băm được; `tuple` thì được.

In [8]:
d = {}
d[(1, 1)] = "ok"
print(d)
try:
    d[[1, 1]] = "lỗi"
except TypeError as e:
    print("TypeError:", e)

{(1, 1): 'ok'}
TypeError: unhashable type: 'list'


## 5. Nền tảng: `defaultdict(list)` để gom nhóm

`defaultdict(list)` tự tạo `[]` cho key chưa có, nên viết `groups[key].append(s)` mà không cần kiểm tra trước.

In [9]:
from collections import defaultdict
groups = defaultdict(list)
for word in ["apple", "avocado", "banana", "blueberry", "cherry"]:
    groups[word[0]].append(word)       # gom theo chữ cái đầu
print(dict(groups))
print(list(groups.values()))

{'a': ['apple', 'avocado'], 'b': ['banana', 'blueberry'], 'c': ['cherry']}
[['apple', 'avocado'], ['banana', 'blueberry'], ['cherry']]


## 6. Lời giải chính — gom nhóm theo dãy hiệu

1. `groups = defaultdict(list)`.
2. Với mỗi `s`: `groups[diff_key(s)].append(s)`.
3. Trả `list(groups.values())`.

In [10]:
def group_by_diff(strings: list[str]) -> list[list[str]]:
    groups = defaultdict(list)
    for s in strings:
        groups[diff_key(s)].append(s)
    return list(groups.values())

print(group_by_diff(ex1))
print(group_by_diff(["a"]))

[['abc', 'bcd', 'xyz'], ['acef'], ['az', 'ba'], ['a', 'z']]
[['a']]


## 7. Trace từng bước

In [11]:
def trace_group(strings):
    groups = defaultdict(list)
    for s in strings:
        key = diff_key(s)
        is_new = key not in groups
        groups[key].append(s)
        print(f"{s:<5} key={str(key):<10} {'nhóm MỚI' if is_new else 'vào nhóm cũ'} -> {groups[key]}")
    return list(groups.values())

print(trace_group(ex1))

abc   key=(1, 1)     nhóm MỚI -> ['abc']
bcd   key=(1, 1)     vào nhóm cũ -> ['abc', 'bcd']
acef  key=(2, 2, 1)  nhóm MỚI -> ['acef']
xyz   key=(1, 1)     vào nhóm cũ -> ['abc', 'bcd', 'xyz']
az    key=(25,)      nhóm MỚI -> ['az']
ba    key=(25,)      vào nhóm cũ -> ['az', 'ba']
a     key=()         nhóm MỚI -> ['a']
z     key=()         vào nhóm cũ -> ['a', 'z']
[['abc', 'bcd', 'xyz'], ['acef'], ['az', 'ba'], ['a', 'z']]


Bảng tóm tắt trạng thái `groups` sau cùng:

| key | nhóm |
|---|---|
| `(1, 1)` | `["abc", "bcd", "xyz"]` |
| `(2, 2, 1)` | `["acef"]` |
| `(25,)` | `["az", "ba"]` |
| `()` | `["a", "z"]` |

Đề cho phép thứ tự bất kỳ, nên output `[["abc","bcd","xyz"],["acef"],["az","ba"],["a","z"]]` cũng đúng như ví dụ đề.

## 8. Cách 3 — Khóa "dịch về bắt đầu bằng `a`"

Một khóa chuẩn hóa khác: dịch mỗi chuỗi sao cho chữ đầu thành `'a'`. Ví dụ `"xyz"` lùi 23 bước → `"abc"`; `"ba"` lùi 1 bước → `"az"`. Cả nhóm sẽ chung một chuỗi đại diện.

In [12]:
def normalize_key(s: str) -> str:
    k = (ord(s[0]) - ord("a")) % 26
    return shift_string(s, -k)

for s in ex1:
    print(f"{s:<5} -> {normalize_key(s)}")

def group_by_normalize(strings: list[str]) -> list[list[str]]:
    groups = defaultdict(list)
    for s in strings:
        groups[normalize_key(s)].append(s)
    return list(groups.values())

print(group_by_normalize(ex1))

abc   -> abc
bcd   -> abc
acef  -> acef
xyz   -> abc
az    -> az
ba    -> az
a     -> a
z     -> a
[['abc', 'bcd', 'xyz'], ['acef'], ['az', 'ba'], ['a', 'z']]


Khóa này dễ đọc khi debug (là một chuỗi thật). Cả hai khóa đều `O(L)` mỗi chuỗi.

## 9. Code nộp LeetCode

In [13]:
from typing import List
from collections import defaultdict

class Solution:
    def groupStrings(self, strings: List[str]) -> List[List[str]]:
        groups = defaultdict(list)
        for s in strings:
            key = tuple((ord(s[i]) - ord(s[i - 1])) % 26 for i in range(1, len(s)))
            groups[key].append(s)
        return list(groups.values())

sol = Solution()
print(sol.groupStrings(["abc", "bcd", "acef", "xyz", "az", "ba", "a", "z"]))
print(sol.groupStrings(["a"]))

[['abc', 'bcd', 'xyz'], ['acef'], ['az', 'ba'], ['a', 'z']]
[['a']]


## 10. Vì sao đúng?

**Bổ đề:** Với `s`, `t` cùng độ dài `L`, `t` là dịch của `s` ⇔ `diff_key(s) == diff_key(t)`.

- (⇒) Nếu `t[i] ≡ s[i] + k (mod 26)` với mọi `i`, thì `t[i] - t[i-1] ≡ s[i] - s[i-1] (mod 26)` → dãy hiệu bằng nhau.
- (⇐) Đặt `k = t[0] - s[0] (mod 26)`. Quy nạp: nếu `t[i-1] ≡ s[i-1] + k` và hiệu bằng nhau thì `t[i] = t[i-1] + d_i ≡ s[i-1] + k + d_i = s[i] + k`.
- Hai chuỗi khác độ dài có dãy hiệu khác độ dài → khác key → không bao giờ bị gộp nhầm.

Vậy "cùng dãy dịch" là một **quan hệ tương đương** và `diff_key` là **bất biến đầy đủ** của nó: gom theo key cho đúng các lớp tương đương.

## 11. Độ phức tạp

`n` = số chuỗi, `L` = độ dài tối đa, `N = tổng độ dài` (≤ `n·L`).

| Cách | Time | Space |
|---|---|---|
| Brute force so với đại diện nhóm | `O(n² · L)` | `O(N)` |
| Hash map + key dãy hiệu | `O(N)` (trung bình, băm tuple dài `L`) | `O(N)` |
| Hash map + key chuẩn hóa về `a` | `O(N)` | `O(N)` |

## 12. Bộ test

Vì output là nhóm theo thứ tự bất kỳ, ta **chuẩn hóa** trước khi so: sắp xếp trong từng nhóm, rồi sắp xếp danh sách nhóm.

In [14]:
def canon(groups):
    return sorted(sorted(g) for g in groups)

def check(name, actual, expected):
    assert canon(actual) == canon(expected), (name, actual, expected)
    print(f"PASS  {name}: {canon(actual)}")

sol = Solution()
check("ví dụ 1", sol.groupStrings(["abc", "bcd", "acef", "xyz", "az", "ba", "a", "z"]),
      [["acef"], ["a", "z"], ["abc", "bcd", "xyz"], ["az", "ba"]])
check("ví dụ 2", sol.groupStrings(["a"]), [["a"]])
check("quay vòng z->a", sol.groupStrings(["za", "ab", "yz"]), [["za", "ab", "yz"]])
check("cùng độ dài khác nhóm", sol.groupStrings(["ab", "ba"]), [["ab"], ["ba"]])
check("chuỗi trùng nhau", sol.groupStrings(["abc", "abc"]), [["abc", "abc"]])
check("khác độ dài", sol.groupStrings(["a", "aa", "aaa"]), [["a"], ["aa"], ["aaa"]])
check("toàn bộ một nhóm", sol.groupStrings([shift_string("hello", k) for k in range(26)]),
      [[shift_string("hello", k) for k in range(26)]])

PASS  ví dụ 1: [['a', 'z'], ['abc', 'bcd', 'xyz'], ['acef'], ['az', 'ba']]
PASS  ví dụ 2: [['a']]
PASS  quay vòng z->a: [['ab', 'yz', 'za']]
PASS  cùng độ dài khác nhóm: [['ab'], ['ba']]
PASS  chuỗi trùng nhau: [['abc', 'abc']]
PASS  khác độ dài: [['a'], ['aa'], ['aaa']]
PASS  toàn bộ một nhóm: [['axeeh', 'byffi', 'czggj', 'dahhk', 'ebiil', 'fcjjm', 'gdkkn', 'hello', 'ifmmp', 'jgnnq', 'khoor', 'lipps', 'mjqqt', 'nkrru', 'olssv', 'pmttw', 'qnuux', 'rovvy', 'spwwz', 'tqxxa', 'uryyb', 'vszzc', 'wtaad', 'xubbe', 'yvccf', 'zwddg']]


In [15]:
import random
rng = random.Random(249)
solvers = [group_bruteforce, group_by_diff, group_by_normalize, Solution().groupStrings]
for _ in range(500):
    bases = ["".join(rng.choice("abc") for _ in range(rng.randint(1, 3))) for _ in range(3)]
    strs = [shift_string(rng.choice(bases), rng.randint(0, 25)) for _ in range(rng.randint(1, 12))]
    outs = [canon(f(strs)) for f in solvers]
    assert all(o == outs[0] for o in outs), (strs, outs)
print("500 bộ ngẫu nhiên: 4 cài đặt cho cùng cách chia nhóm — PASS")

500 bộ ngẫu nhiên: 4 cài đặt cho cùng cách chia nhóm — PASS


## 13. Lỗi người mới hay gặp

### Lỗi 1: Không lấy `% 26` cho hiệu

In [16]:
def wrong_no_mod(strings):
    groups = defaultdict(list)
    for s in strings:
        groups[tuple(ord(s[i]) - ord(s[i - 1]) for i in range(1, len(s)))].append(s)
    return list(groups.values())

print("sai :", wrong_no_mod(["az", "ba"]))
print("đúng:", Solution().groupStrings(["az", "ba"]))

sai : [['az'], ['ba']]
đúng: [['az', 'ba']]


**Vì sao sai:** `"az"` có hiệu `25`, `"ba"` có hiệu `-1` — cùng là "lùi 1 = tiến 25" nhưng khác số nếu không quy về `mod 26`. **Sửa:** `% 26`.

### Lỗi 2: Dùng **list** làm key

`groups[[1, 1]]` → `TypeError: unhashable type: 'list'` (đã thấy ở mục 4). **Sửa:** `tuple(...)`.

### Lỗi 3: Ghép dãy hiệu thành chuỗi số không có dấu phân cách

In [17]:
def wrong_concat_key(s):
    return "".join(str((ord(s[i]) - ord(s[i - 1])) % 26) for i in range(1, len(s)))

a, b = "abm", "alm"   # hiệu (1, 11) và (11, 1)
print(diff_key(a), diff_key(b), "| key nối chuỗi:", wrong_concat_key(a), wrong_concat_key(b))

(1, 11) (11, 1) | key nối chuỗi: 111 111


**Vì sao sai:** `(1, 11)` và `(11, 1)` đều nối thành `"111"` → hai nhóm khác nhau bị gộp. **Sửa:** dùng `tuple`, hoặc nối có dấu phân cách `",".join(...)`.

### Lỗi 4: Quên chuỗi một ký tự
Với `len(s) == 1`, `range(1, 1)` rỗng → key `()`. Mọi chuỗi một ký tự (`"a"`, `"z"`...) cùng một nhóm — đúng theo đề. Nếu code truy cập `s[1]` mà không kiểm tra độ dài sẽ `IndexError`.

### Lỗi 5: Dùng `normalize_key` nhưng tính dịch sai chiều
Viết `shift_string(s, k)` thay vì `shift_string(s, -k)` sẽ đẩy chữ đầu **ra xa** `a` (thành `2k`), các chuỗi cùng nhóm không còn chung key.

## 14. Mẹo nhận dạng pattern

- "Gom nhóm các phần tử tương đương" → tìm **khóa chuẩn hóa (canonical form)** + **hash map** (`defaultdict(list)`).
- Ví dụ quen thuộc: Group Anagrams (khóa = chuỗi đã sắp xếp/mảng đếm), Group Shifted Strings (khóa = dãy hiệu mod 26), Isomorphic patterns (khóa = thứ tự xuất hiện lần đầu).
- Biến đổi "cộng cùng một hằng số cho mọi phần tử" → **hiệu liên tiếp** là bất biến.
- "Vòng tròn / quay vòng" → nghĩ tới `% m`.

## 15. Biến thể

In [18]:
def pattern_key(word: str) -> tuple:
    """Khóa 'thứ tự xuất hiện lần đầu': 'egg' -> (0,1,1), 'add' -> (0,1,1) (Find and Replace Pattern, LC 890)."""
    first = {}
    return tuple(first.setdefault(ch, len(first)) for ch in word)

def find_and_replace_pattern(words, pattern):
    pk = pattern_key(pattern)
    return [w for w in words if pattern_key(w) == pk]

print(pattern_key("egg"), pattern_key("add"), pattern_key("foo"), pattern_key("bar"))
print(find_and_replace_pattern(["abc", "deq", "mee", "aqq", "dkd", "ccc"], "abb"))

(0, 1, 1) (0, 1, 1) (0, 1, 1) (0, 1, 2)
['mee', 'aqq']


`first.setdefault(ch, len(first))`: nếu `ch` chưa có thì gán `first[ch] = len(first)` (số thứ tự mới) và trả về giá trị đó; nếu có rồi thì trả giá trị cũ.

## 16. Bài tập tự luyện (TODO)

### Bài 1 — Tính tay
Tính `diff_key` của `"bdf"`, `"zbd"`, `"ace"`, `"abd"`. Chuỗi nào cùng nhóm?

### Bài 2 — Viết `shift_string`

In [19]:
def exercise_shift(s: str, k: int) -> str:
    # TODO: ord - ord('a'), cộng k, % 26, chr
    pass

# assert exercise_shift("xyz", 3) == "abc"
# assert exercise_shift("abc", -1) == "zab"

### Bài 3 — Tự viết `groupStrings` bằng key dãy hiệu

In [20]:
def exercise_group(strings: list[str]) -> list[list[str]]:
    # TODO: defaultdict(list); key = tuple(hiệu % 26); return list(values)
    pass

# assert canon(exercise_group(["abc", "bcd", "az", "ba", "a"])) == [["a"], ["abc", "bcd"], ["az", "ba"]]

### Bài 4 — Số bước dịch giữa hai chuỗi
Trả về `k` nhỏ nhất (0..25) sao cho dịch `s` đi `k` bước được `t`; nếu không thể trả `-1`.

In [21]:
def exercise_shift_amount(s: str, t: str) -> int:
    # TODO: kiểm tra độ dài; k = (t[0] - s[0]) % 26; xác nhận lại
    pass

# assert exercise_shift_amount("abc", "xyz") == 23
# assert exercise_shift_amount("ab", "ba") == -1

### Bài 5 — Nhóm lớn nhất
Trả về **kích thước** của nhóm lớn nhất. Ví dụ ví dụ 1 → `3`.

In [22]:
def exercise_largest_group(strings: list[str]) -> int:
    # TODO: Counter theo key
    pass

# assert exercise_largest_group(["abc", "bcd", "acef", "xyz", "az", "ba", "a", "z"]) == 3

### Bài 6 — Mini challenge: mỗi nhóm sắp theo "số bước dịch từ đại diện"
Với mỗi nhóm, sắp xếp các chuỗi theo `k` = số bước dịch từ chuỗi chuẩn hóa (bắt đầu bằng `a`) tới chuỗi đó. Ví dụ nhóm `["xyz", "abc", "bcd"]` → `["abc", "bcd", "xyz"]` (k = 0, 1, 23).

In [23]:
def exercise_sorted_groups(strings: list[str]) -> list[list[str]]:
    # TODO: gom theo normalize_key; sort trong nhóm theo (ord(s[0]) - 97)
    pass

# assert canon(exercise_sorted_groups(["xyz", "abc", "bcd"])) == [["abc", "bcd", "xyz"]]
# assert exercise_sorted_groups(["xyz", "abc", "bcd"])[0] == ["abc", "bcd", "xyz"]

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Hiệu từng cặp kề, mod 26 | `"zbd"`: b - z = 2 | `bdf, zbd, ace` đều `(2, 2)` |
| 2 | Mục 2 | `% 26` xử lý cả `k` âm | `chr((ord(c) - 97 + k) % 26 + 97)` |
| 3 | Mục 6 | `tuple(... for i in range(1, len(s)))` | `list(groups.values())` |
| 4 | `k` do chữ đầu quyết định | `(ord(t[0]) - ord(s[0])) % 26` | so `shift(s, k) == t` |
| 5 | Đếm số phần tử mỗi key | `Counter(diff_key(s) for s in strings)` | `max(counter.values())` |
| 6 | Đại diện bắt đầu bằng `a` | `k = (ord(s[0]) - 97) % 26` | `sorted(g, key=lambda s: ...)` |

## 18. Đáp án tham khảo, template và checklist

Bài 1: `"bdf"` → (2, 2); `"zbd"` → ((98-122)%26 = 2, 2) = (2, 2); `"ace"` → (2, 2); `"abd"` → (1, 2). Ba chuỗi đầu cùng nhóm, `"abd"` riêng.

In [24]:
from collections import Counter

def answer_shift(s, k):
    return "".join(chr((ord(c) - 97 + k) % 26 + 97) for c in s)

def answer_group(strings):
    groups = defaultdict(list)
    for s in strings:
        groups[tuple((ord(s[i]) - ord(s[i - 1])) % 26 for i in range(1, len(s)))].append(s)
    return list(groups.values())

def answer_shift_amount(s, t):
    if len(s) != len(t):
        return -1
    k = (ord(t[0]) - ord(s[0])) % 26
    return k if answer_shift(s, k) == t else -1

def answer_largest_group(strings):
    return max(Counter(diff_key(s) for s in strings).values())

def answer_sorted_groups(strings):
    groups = defaultdict(list)
    for s in strings:
        groups[normalize_key(s)].append(s)
    return [sorted(g, key=lambda s: (ord(s[0]) - 97) % 26) for g in groups.values()]

print([diff_key(x) for x in ["bdf", "zbd", "ace", "abd"]])
assert answer_shift("xyz", 3) == "abc" and answer_shift("abc", -1) == "zab"
assert canon(answer_group(["abc", "bcd", "az", "ba", "a"])) == [["a"], ["abc", "bcd"], ["az", "ba"]]
assert answer_shift_amount("abc", "xyz") == 23 and answer_shift_amount("ab", "ba") == -1
assert answer_shift_amount("a", "abc") == -1
assert answer_largest_group(["abc", "bcd", "acef", "xyz", "az", "ba", "a", "z"]) == 3
assert answer_sorted_groups(["xyz", "abc", "bcd"])[0] == ["abc", "bcd", "xyz"]
for _ in range(300):
    s = "".join(rng.choice("abcxyz") for _ in range(rng.randint(1, 4)))
    t = "".join(rng.choice("abcxyz") for _ in range(rng.randint(1, 4)))
    assert (answer_shift_amount(s, t) != -1) == same_group_try_all(s, t)
print("Đáp án tham khảo: tất cả test PASS")

[(2, 2), (2, 2), (2, 2), (1, 2)]
Đáp án tham khảo: tất cả test PASS


### Template nhớ nhanh

```python
groups = defaultdict(list)
for s in strings:
    key = tuple((ord(s[i]) - ord(s[i-1])) % 26 for i in range(1, len(s)))
    groups[key].append(s)
return list(groups.values())
```

### Checklist submit

- [ ] Hiệu lấy `% 26` để xử lý quay vòng `z → a`.
- [ ] Key là `tuple` (hashable), không phải `list`, không nối chuỗi thiếu dấu phân cách.
- [ ] Chuỗi một ký tự cho key `()` và cùng một nhóm.
- [ ] Trả `list(groups.values())` — thứ tự nhóm tùy ý.
- [ ] `from typing import List` và `from collections import defaultdict` khi chạy ngoài LeetCode.